[this thread](https://stackoverflow.com/questions/13786210/list-all-files-in-an-online-directory-with-python)

https://www1.nyc.gov/assets/planning/download/zip/data-maps/open-data/nyc_mappluto_20v8_arc_shp.zip

In [1]:
import httplib2
import bs4 as bs
from bs4 import BeautifulSoup, SoupStrainer
import pandas as pd

In [2]:
archive_link = 'https://www1.nyc.gov/site/planning/data-maps/open-data/bytes-archive.page?sorts[year]=0'

http = httplib2.Http()
status, response = http.request(archive_link)
l = []

In [3]:
for link in bs.BeautifulSoup(response, 'html.parser',
                             parse_only=SoupStrainer('a')):
    if link.has_attr('href'):
        l.append(link['href'])

In [4]:
links = pd.DataFrame(l, columns = ['links'])

In [5]:
links = links[links.links.str.match('.+mappluto.+zip$')]
links = links[links.links.str.contains('fgdb')==False].reset_index(drop = True)

In [6]:
zips = links.links.str.split('open-data/', expand = True).loc[:,1]

In [7]:
links['zips'] = zips

In [8]:
year_regexp = '(?<=mappluto_)(.{2})'
year = '20' + links.zips.str.extract(year_regexp)

In [9]:
links['year'] = year

In [10]:
version_regexp = '(?<=pluto_\d{2}v)(.{3})'

versions = links.zips.str.extract(version_regexp).iloc[:,0]
versions[versions.isnull()] = '1'
versions[versions.str.match('\d_\d')] = versions.str.replace('_', '.')
versions = versions.str.replace('([a-z]|_)', '')
links['versions'] = pd.to_numeric(versions)


In [20]:
links = links.sort_values(['year', 'versions'])
links = links[links.versions == links.groupby(['year'])['versions'].transform(max)].reset_index(drop = True)
links['full_path'] = 'https://www1.nyc.gov' + links.links

In [25]:
links.to_csv('pluto_zips.csv', index = False)